# MILK10k setup: tables and folders

This notebook creates the MILK10k dataset-specific Bronze tables and (later) Gold tables, plus their storage folders. The shared Silver tables are created once in `notebooks/00_setup_storage_and_shared_tables.ipynb`, not here.

Run this after `notebooks/00_setup_storage_and_shared_tables.ipynb` and before `10_bronze_ingest.ipynb`. It also prints the exact landing paths where the source archive must exist.


In [ ]:
%run ../_setup_env


In [ ]:
from data_platform.layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "milk10k.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)

print(layout)
for archive in ARCHIVES:
    print(f"{archive['source_split']} archive upload path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image rows table: {bronze_tables['image_blobs']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Create Bronze Objects

Create the MILK10k Bronze source metadata table, image blob table, and dataset-specific landing and metadata folders.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.milk10k_image_blobs (
      image_id STRING,
      source_split STRING,
      archive_member_path STRING,
      source_archive_uri STRING,
      image_bytes BINARY,
      byte_length BIGINT,
      source_checksum STRING
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.milk10k_source_metadata (
      image_id STRING,
      source_split STRING,
      source_uri STRING,
      attribution STRING,
      copyright_license STRING,
      age_approx STRING,
      anatom_site_1 STRING,
      anatom_site_2 STRING,
      anatom_site_special STRING,
      concomitant_biopsy STRING,
      diagnosis_1 STRING,
      diagnosis_2 STRING,
      diagnosis_3 STRING,
      diagnosis_4 STRING,
      diagnosis_confirm_type STRING,
      image_manipulation STRING,
      image_type STRING,
      lesion_id STRING,
      melanocytic STRING,
      sex STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)
dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)
for archive in ARCHIVES:
    dbutils.fs.mkdirs(archive["metadata_dir_path"])

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Gold

`gold.manifest_rows` and `gold.manifest_registry` are shared, not dataset-specific — created once in `notebooks/00_setup_storage_and_shared_tables.ipynb`, same reasoning as Silver. Nothing to do in this notebook. See `docs/project-checklist.md` for what's built so far (a `sample-v1` release) versus deferred (the full-scale release, dataset card generation).


## Upload Checklist

Upload the archive listed in `config/datasets/milk10k.yaml` to the path printed above before running `10_bronze_ingest.ipynb`.
